# Non-Rigid Serial-Slice Alignment

Run a selected non-rigid CS13 slice-pair alignment and compare full-data with 10,000-cell reference alignment.

This curated notebook targets the current Dynamo-free Spateo API. Edit the configuration cell before execution.


## Configure the slice pair


In [ ]:
import os
import warnings
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import numpy as np
import spateo as st
import torch

warnings.filterwarnings("ignore")
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print(f"Spateo {st.__version__}; alignment device: {DEVICE}")

import anndata as ad

SLICE_PATHS = [
    Path("/DATA/User/gaomohan/DATA/CS13_Project/cs13/add_celltype/adata_109_processed.h5ad"),
    Path("/DATA/User/gaomohan/DATA/CS13_Project/cs13/add_celltype/adata_118_processed.h5ad"),
]
SPATIAL_KEY = "spatial"
ANNOTATION_KEY = "celltype"
ALIGN_KEY = "align_spatial"
WRITE_ANIMATION = False
ANIMATION_PATH = Path("/DATA/User/gaomohan/Alignment/results/nonrigid_alignment")


## Load and preprocess the pair


In [ ]:
slices = [st.read_h5ad(path) for path in SLICE_PATHS]
for path, adata in zip(SLICE_PATHS, slices):
    if SPATIAL_KEY not in adata.obsm or np.asarray(adata.obsm[SPATIAL_KEY]).shape[1] != 2:
        raise ValueError(f"{path.name} requires two-dimensional spatial coordinates.")
    if ANNOTATION_KEY not in adata.obs:
        raise KeyError(f"{path.name} is missing obs[{ANNOTATION_KEY!r}].")


def preprocess_slice(adata):
    adata = st.pp.filter_cells(adata, min_expr_genes=10, inplace=False)
    adata = st.pp.filter_genes(adata, min_cells=3, inplace=False)
    if "counts_X" not in adata.layers:
        source_layer = next(
            (candidate for candidate in ("counts", "raw_counts") if candidate in adata.layers),
            None,
        )
        if source_layer is None:
            warnings.warn("No count layer found; treating X as counts. Verify this assumption.")
        adata.layers["counts_X"] = (
            adata.layers[source_layer].copy() if source_layer is not None else adata.X.copy()
        )
    st.pp.normalize_total(
        adata,
        layer="counts_X",
        out_layer="norm_X",
        target_sum=None,
        size_factor_key="Size_Factor",
        inplace=True,
    )
    st.pp.log1p_layer(
        adata,
        layer="norm_X",
        out_layer="log1p_X",
        set_X=True,
        inplace=True,
    )
    return adata


slices = [preprocess_slice(adata) for adata in slices]
st.align.group_pca(slices, pca_key="X_pca", use_hvg=False)


## Run the selected non-rigid model


In [ ]:
aligned_slices, mapping = st.align.morpho_align(
    models=[adata.copy() for adata in slices],
    rep_layer="X_pca",
    rep_field="obsm",
    dissimilarity="cos",
    spatial_key=SPATIAL_KEY,
    key_added=ALIGN_KEY,
    device=DEVICE,
    verbose=True,
    beta=1,
    lambdaVF=1,
    max_iter=300,
    K=100,
)

st.pl.overlay_slices_2d(
    slices=aligned_slices,
    spatial_key=f"{ALIGN_KEY}_nonrigid",
    height=3,
    overlay_type="backward",
    show_legend=False,
)


## Optionally export the optimization animation


In [ ]:
if WRITE_ANIMATION:
    ANIMATION_PATH.parent.mkdir(parents=True, exist_ok=True)
    st.pl.optimization_animation(
        aligned_slices=aligned_slices,
        spatial_key=SPATIAL_KEY,
        key_added=ALIGN_KEY,
        iter_key_added="iter_spatial",
        filename=str(ANIMATION_PATH),
        fps=10,
        stepsize=10,
        label_key=ANNOTATION_KEY,
    )


## Compare full-data and 10,000-cell reference alignment


In [ ]:
aligned_full, _ = st.align.morpho_align(
    models=[adata.copy() for adata in slices],
    rep_layer="X_pca",
    rep_field="obsm",
    dissimilarity="cos",
    spatial_key=SPATIAL_KEY,
    key_added=ALIGN_KEY,
    device=DEVICE,
    verbose=True,
    beta=1,
    lambdaVF=1,
    K=50,
)

aligned_down, aligned_reference, _, _ = st.align.morpho_align_ref(
    models=[adata.copy() for adata in slices],
    n_sampling=10000,
    sampling_method="random",
    rep_layer="X_pca",
    rep_field="obsm",
    dissimilarity="cos",
    spatial_key=SPATIAL_KEY,
    key_added=ALIGN_KEY,
    device=DEVICE,
    verbose=True,
    beta=1,
    lambdaVF=1,
    K=50,
)


In [ ]:
def combine_pair(models, label):
    combined = ad.concat(models, label="batch", keys=["slice_3", "slice_4"])
    combined.obs["alignment_source"] = label
    return combined


comparison = [
    combine_pair(aligned_down, "10,000-cell reference"),
    combine_pair(aligned_full, "full data"),
]
st.pl.slices_2d(
    slices=comparison,
    slices_key="alignment_source",
    label_key="batch",
    spatial_key=f"{ALIGN_KEY}_nonrigid",
    height=4,
    center_coordinate=False,
    show_legend=False,
)
